### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/00_Development.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Fnotebooks%2Fthesis%2F00_Development.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Fnotebooks%2Fthesis%2F00_Development.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/00_Development.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.

Confirmation and collection notebooks also require the campaign artifacts prepared in notebook **01**.

See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.


In [ ]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME,
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "notebooks" / "init.py" for path in _locations
                     if (path / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, revision=REVISION, 
    runtime=RUNTIME, cuda=CUDA
)

# 00 — Check learning and cost before freezing

Run **one seed-17 validation stream**. Start with CIFAR-10/platform, then learned; repeat for CIFAR-100. These are development observations, not confirmation results.

Select a **TensorFlow 2.20 / Keras 3** kernel, restart the kernel, then **Run All**.

In [ ]:
from notebooks.thesis.workflow import check_runtime


print(check_runtime())
from IPython import get_ipython


get_ipython().run_line_magic("matplotlib", "inline")
from common.dataloader import get_datasets
from common.model import get_model
from common.train import train_model
from notebooks.thesis.workflow import load_development, attach_route, close_run, finish_run
from notebooks.thesis.presentation import describe_run, show_learning_results, show_diagnostics, show_saved_replay

### 1. Select one stream

Only DATASET and CONDITION normally need changing. The [recipe rationale](HYPERPARAMETER_RATIONALE.md) explains the central YAML.

In [ ]:
DATASET = "cifar10"  # "cifar10" or "cifar100"
CONDITION = "baseline"  # Start here, then "learned" in a fresh kernel.
SEED = 17
config, context = load_development(ROOT / "notebooks/thesis/configs" / f"{DATASET}.yaml",
                                   condition=CONDITION, seed=SEED)
describe_run(config)

### 2. Load data and create the model

The common APIs own splitting, replay and class growth. The route attaches to this same model.

In [ ]:
project = config.common
trainset, valset = get_datasets(project)
bundle = get_model(project)
attach_route(context, bundle)

### 3. Train once

After an interruption, restart the kernel and **Run All** to resume the same stream. Work after the latest valid checkpoint is repeated.

In [ ]:
if context.get("training_started"):
    raise RuntimeError("Restart the kernel before training another stream.")
context["training_started"] = True
try:
    history = train_model(project, bundle, trainset, valset=valset)
except BaseException:
    close_run(context, release=True)
    raise
finally:
    close_run(context)

### 4. Save and read the results

Accuracy is percent; forgetting and backward transfer are signed percentage points. These are validation observations for development.

In [ ]:
evaluations = finish_run(context, config, bundle, history, trainset, valset)
RUN, VIEW_DIR = show_learning_results(config, bundle)

### 5. Review phases, cost and saved replay

Read the full stream before freezing: new-class learning, old-class retention, phase changes, gate visits and late-task cost. Replay labels are generation conditions, not verified image semantics.

In [ ]:
review = show_diagnostics(RUN, VIEW_DIR)
show_saved_replay(config, bundle, RUN, VIEW_DIR)

Before freezing, inspect new-class learning against the plotted all-seen chance level, old-class retention, replay, phase deltas and gate visits. Check the full ten-task CIFAR-100 run for late-task cost and memory. Tune only the central YAML if development evidence warrants it, record why, and rerun the necessary stream. Software smoke checks do not establish useful learning. Once the recipe is adequate, run **01_Freeze_Experiment.ipynb**.